# Processing CESM2-LENS PiC data

### Set up
#### Packages

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
from scipy import stats
import warnings
warnings.simplefilter('ignore', UserWarning)
warnings.filterwarnings('ignore')
import datetime as dt
from datetime import timedelta
from Processing_functions import FixLongitude, FixTime, FixGrid, InterPlevels
xr.set_options(display_expand_data=False);
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client
from dask.diagnostics import ProgressBar
import random
import os

#### Filepaths & name variables

In [2]:
## File name
cesm2piC = 'b.e21.B1850.f09_g17.CMIP6-piControl.001'

## Filepaths
path_to_arch = "/glade/campaign/collections/cmip/CMIP6/timeseries-cmip6/"
comp = 'atm'
freq = 0 # 0: monthly, 1: daily
var_ind = 7

# ATM
# 7,   8, 9, 11,     12,     13, 14,     15,       16,      17,     18,     19,    20
# PSL, U, V, TREFHT, RESTOM, Z3, FSNTOA, TGCLDLWP, AEROD_v, PRECSC, PRECSL, PRECC, PRECL

# ICE
# 0,    1,  2,     3,     4,      5
# aice, hi, meltt, meltb, congel, frazil

# OCN
# 0
# MOC

# Variables
var_list = {'atm': ['TS','FLDS','CLOUD','FLNS','FSNS','FLNT','FSNT','PSL','U','V','T','TREFHT',
                    'RESTOM','Z3','FSNTOA','TGCLDLWP','AEROD_v',
                    'PRECSC','PRECSL','PRECC','PRECL'],
            'ice': ['aice','hi','meltt','meltb','congel','frazil'],
            'ocn': ['MOC','TEMP']}
var_ext = {0: '', 1: '_d'}
var = var_list[comp][var_ind]+var_ext[freq]

# Extensions
h_ext = {'atm': ['.h0.'],
       'ice': ['.h.','.h1.'],
       'ocn': ['.h.']}
mod_com = {'atm': 'cam',
           'ice': 'cice',
           'ocn': 'pop'}
time_path = {'atm': ['month_1'],
                'ice': ['month_1','day_1'],
                'ocn': ['month_1']}
vert_lev = {'atm': [False,False,True,False,False,False,False,False,True,True,True,
                    False,False,True,False,False,False,
                    False,False,False,False],
            'ice': [False,False,False,False,False,False],
            'ocn': [False,False]}

path_to_outdata = '/glade/work/glydia/processed_CESM2_LENS_data/longer_recent/'
plot_levels = [300,500,850,925]

In [3]:
cluster = PBSCluster(cores    = 1,
                     memory   = '25GiB',
                     queue    = 'casper',
                     walltime = '04:00:00',
                     account  = 'UCUB0137',
                     log_directory = '/glade/u/home/glydia/python/logs/',
                     name='piControl_'+var)
cluster.scale(4*9)
client = Client(cluster)

In [4]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/glydia/Arctic_breakdown/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/glydia/Arctic_breakdown/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.117:40699,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/glydia/Arctic_breakdown/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [5]:
## Chunking variables
time_ch = 365*2 if freq == 1 else 600
chunks = {
    'atm': {'time': time_ch, 'lat': 96, 'lon': 144, 'lev': -1},
    'ice': {'time': time_ch, 'nj': 192, 'ni': 160},
    'ocn': {'time': time_ch, 'nlat': 64, 'nlon': 96, 'z_t': 5}
}

### Load & modify data
#### Control data

In [6]:
startyrs = np.array([501,
 797,
 1328,
 1776,
 460,
 739,
 778,
 836,
 965,
 485,
 159,
 264,
 760,
 808,
 1162,
 990,
 478,
 221,
 1606,
 1550,
 431,
 710,
 212,
 635,
 1577,
 578,
 8,
 203,
 796,
 919,
 1682,
 10,
 1700,
 1068,
 918,
 1292,
 1293,
 1479,
 1574,
 1095,
 394,
 1718,
 795,
 1132,
 1620,
 129,
 1906,
 1034,
 175,
 1405,
 1621])

In [7]:
not_vert = not vert_lev[comp][var_ind]

In [8]:
slice_numbers = np.arange(1,52)

In [9]:
%%time
if var != 'RESTOM' and not_vert:
    filepath = path_to_arch+cesm2piC+'/'+comp+'/proc/tseries/'+time_path[comp][freq]+'/'
    num = 0
    for startyr in startyrs:
        yr_range = np.array([str(i).zfill(4) for i in np.arange(startyr,startyr+76)])
        
        endyr = startyr+74
        start_floor = np.floor_divide(startyr,100)
        end_floor = np.floor_divide(endyr,100)
        
        if start_floor == end_floor:
            if start_floor == 0:
                instart = str(1).zfill(4)
                inend = str(99).zfill(4)
            elif start_floor == 19:
                instart = str(1900).zfill(4)
                inend = str(2000).zfill(4)
            else:
                instart = str(start_floor*100).zfill(4)
                inend = str(start_floor*100+99).zfill(4)
        
            yr_extn = {'in': ["."+instart+"01-"+inend+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}

            outpath = path_to_outdata+cesm2piC+h_ext[comp][freq]+var+yr_extn['out'][freq]+'zarr'
            if os.path.isdir(outpath):
                print(cesm2piC+h_ext[comp][freq]+var+yr_extn['out'][freq]+'zarr exists')
                continue
            else:
                # Open dataset
                filename = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr_extn['in'][freq]+'nc'
                print(filepath+filename)
                ds = dask.delayed(xr.open_dataset)(filepath+filename,chunks=chunks[comp])
                # ds = xr.open_dataset(filepath+filename,chunks=chunks[comp])
        else:
            if start_floor == 0:
                instart1 = str(1).zfill(4)
                inend1 = str(99).zfill(4)
            else:
                instart1 = str(start_floor*100).zfill(4)
                inend1 = str(start_floor*100+99).zfill(4)
                
            if end_floor == 19:
                instart2 = str(1900).zfill(4)
                inend2 = str(2000).zfill(4)
            else:
                instart2 = str(end_floor*100).zfill(4)
                inend2 = str(end_floor*100+99).zfill(4)
    
            yr_extn1 = {'in': ["."+instart1+"01-"+inend1+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
    
            yr_extn2 = {'in': ["."+instart2+"01-"+inend2+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
            yr_extn = {'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}

            # Open dataset
            filename1 = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr_extn1['in'][freq]+'nc'
            filename2 = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr_extn2['in'][freq]+'nc'
            print(filepath+filename1)
            print(filepath+filename2)
            ds = dask.delayed(xr.open_mfdataset)([filepath+filename1, filepath+filename2],chunks=chunks[comp])
            # ds = xr.open_mfdataset([filepath+filename1, filepath+filename2],chunks=chunks[comp])

        
        dsv = ds[var]
     
        #del ds
        
        processed_list = []
        for i in range(0,len(yr_range)-1):
            # If monthly data
            if freq == 0:
                startyr_sl = yr_range[i]
                endyr_sl = yr_range[i+1]
                ann_slice = dsv.sel(time=slice(startyr_sl+'-02-01',endyr_sl+'-01-17')) #if add_cyclic else dsv.sel(time=slice(startyr+'-02-10',endyr+'-01-17'),lat=slice(0,90))
                print('sliced '+startyr_sl+'-02-01 to '+endyr_sl+'-01-17')
                
                #fixedtime_data = dask.delayed(FixTime)(ann_slice)
                fixedtime_data = FixTime(ann_slice)
                print('   fixed time')
        
                if comp == 'ice':
                    fixedgrid_data = dask.delayed(FixGrid)(fixedtime_data,'ice','gx1v7')
                    processed_list.append(fixedgrid_data)
                    print('   fixed CICE grid')
        
                    #del ann_slice, fixedtime_data, fixedgrid_data
                elif comp == 'ocn':
                    if var == 'TEMP':
                        fixedtime_data = fixedtime_data.sel(z_t=0,method='nearest')
                        fixedgrid_data = dask.delayed(FixGrid)(fixedtime_data,'ocn','gx1v7')
                        processed_list.append(fixedgrid_data)
                        print('   fixed CICE grid')
                    else:
                        processed_list.append(fixedtime_data)
        
                    #del ann_slice, fixedtime_data
                else:
                    fixedgrid_data = dask.delayed(FixLongitude)(fixedtime_data, False)
                    processed_list.append(fixedgrid_data)
                    print('   fixed longitude')
        
                    #del ann_slice, fixedtime_data, fixedgrid_data
                    
        
        if freq == 0 and not_vert:
            processed_comp = dask.compute(*processed_list)
            print('computed list')
            
            processed_out = xr.concat(processed_comp,dim='time').chunk({'time':111})
            print('concatenated data')
    
            processed_out = processed_out.expand_dims(slice=[slice_numbers[num]])
            
            processed_out.to_netcdf(path_to_outdata+cesm2piC+h_ext[comp][freq]+var+yr_extn['out'][freq]+'nc', 
                                        format='NETCDF4',encoding={var: {"zlib": True, "complevel": 1}})
            print('wrote data to disk')
    
        num = num+1

/glade/campaign/collections/cmip/CMIP6/timeseries-cmip6/b.e21.B1850.f09_g17.CMIP6-piControl.001/atm/proc/tseries/month_1/b.e21.B1850.f09_g17.CMIP6-piControl.001.cam.h0.PSL.050001-059912.nc
sliced 0501-02-01 to 0502-01-17
   fixed time
   fixed longitude
sliced 0502-02-01 to 0503-01-17
   fixed time
   fixed longitude
sliced 0503-02-01 to 0504-01-17
   fixed time
   fixed longitude
sliced 0504-02-01 to 0505-01-17
   fixed time
   fixed longitude
sliced 0505-02-01 to 0506-01-17
   fixed time
   fixed longitude
sliced 0506-02-01 to 0507-01-17
   fixed time
   fixed longitude
sliced 0507-02-01 to 0508-01-17
   fixed time
   fixed longitude
sliced 0508-02-01 to 0509-01-17
   fixed time
   fixed longitude
sliced 0509-02-01 to 0510-01-17
   fixed time
   fixed longitude
sliced 0510-02-01 to 0511-01-17
   fixed time
   fixed longitude
sliced 0511-02-01 to 0512-01-17
   fixed time
   fixed longitude
sliced 0512-02-01 to 0513-01-17
   fixed time
   fixed longitude
sliced 0513-02-01 to 0514-01-17

HDF5-DIAG: Error detected in HDF5 (1.14.6) MPI-process 0:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/glade/work/gl

wrote data to disk
/glade/campaign/collections/cmip/CMIP6/timeseries-cmip6/b.e21.B1850.f09_g17.CMIP6-piControl.001/atm/proc/tseries/month_1/b.e21.B1850.f09_g17.CMIP6-piControl.001.cam.h0.PSL.070001-079912.nc
/glade/campaign/collections/cmip/CMIP6/timeseries-cmip6/b.e21.B1850.f09_g17.CMIP6-piControl.001/atm/proc/tseries/month_1/b.e21.B1850.f09_g17.CMIP6-piControl.001.cam.h0.PSL.080001-089912.nc
sliced 0797-02-01 to 0798-01-17
   fixed time
   fixed longitude
sliced 0798-02-01 to 0799-01-17
   fixed time
   fixed longitude
sliced 0799-02-01 to 0800-01-17
   fixed time
   fixed longitude
sliced 0800-02-01 to 0801-01-17
   fixed time
   fixed longitude
sliced 0801-02-01 to 0802-01-17
   fixed time
   fixed longitude
sliced 0802-02-01 to 0803-01-17
   fixed time
   fixed longitude
sliced 0803-02-01 to 0804-01-17
   fixed time
   fixed longitude
sliced 0804-02-01 to 0805-01-17
   fixed time
   fixed longitude
sliced 0805-02-01 to 0806-01-17
   fixed time
   fixed longitude
sliced 0806-02-01 

HDF5-DIAG: Error detected in HDF5 (1.14.6) MPI-process 0:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/glade/work/gl

wrote data to disk
/glade/campaign/collections/cmip/CMIP6/timeseries-cmip6/b.e21.B1850.f09_g17.CMIP6-piControl.001/atm/proc/tseries/month_1/b.e21.B1850.f09_g17.CMIP6-piControl.001.cam.h0.PSL.130001-139912.nc
/glade/campaign/collections/cmip/CMIP6/timeseries-cmip6/b.e21.B1850.f09_g17.CMIP6-piControl.001/atm/proc/tseries/month_1/b.e21.B1850.f09_g17.CMIP6-piControl.001.cam.h0.PSL.140001-149912.nc
sliced 1328-02-01 to 1329-01-17
   fixed time
   fixed longitude
sliced 1329-02-01 to 1330-01-17
   fixed time
   fixed longitude
sliced 1330-02-01 to 1331-01-17
   fixed time
   fixed longitude
sliced 1331-02-01 to 1332-01-17
   fixed time
   fixed longitude
sliced 1332-02-01 to 1333-01-17
   fixed time
   fixed longitude
sliced 1333-02-01 to 1334-01-17
   fixed time
   fixed longitude
sliced 1334-02-01 to 1335-01-17
   fixed time
   fixed longitude
sliced 1335-02-01 to 1336-01-17
   fixed time
   fixed longitude
sliced 1336-02-01 to 1337-01-17
   fixed time
   fixed longitude
sliced 1337-02-01 

...

wrote data to disk
/glade/campaign/collections/cmip/CMIP6/timeseries-cmip6/b.e21.B1850.f09_g17.CMIP6-piControl.001/atm/proc/tseries/month_1/b.e21.B1850.f09_g17.CMIP6-piControl.001.cam.h0.PSL.140001-149912.nc
sliced 1405-02-01 to 1406-01-17
   fixed time
   fixed longitude
sliced 1406-02-01 to 1407-01-17
   fixed time
   fixed longitude
sliced 1407-02-01 to 1408-01-17
   fixed time
   fixed longitude
sliced 1408-02-01 to 1409-01-17
   fixed time
   fixed longitude
sliced 1409-02-01 to 1410-01-17
   fixed time
   fixed longitude
sliced 1410-02-01 to 1411-01-17
   fixed time
   fixed longitude
sliced 1411-02-01 to 1412-01-17
   fixed time
   fixed longitude
sliced 1412-02-01 to 1413-01-17
   fixed time
   fixed longitude
sliced 1413-02-01 to 1414-01-17
   fixed time
   fixed longitude
sliced 1414-02-01 to 1415-01-17
   fixed time
   fixed longitude
sliced 1415-02-01 to 1416-01-17
   fixed time
   fixed longitude
sliced 1416-02-01 to 1417-01-17
   fixed time
   fixed longitude
sliced 1417-

HDF5-DIAG: Error detected in HDF5 (1.14.6) MPI-process 0:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/glade/work/gl

wrote data to disk
/glade/campaign/collections/cmip/CMIP6/timeseries-cmip6/b.e21.B1850.f09_g17.CMIP6-piControl.001/atm/proc/tseries/month_1/b.e21.B1850.f09_g17.CMIP6-piControl.001.cam.h0.PSL.160001-169912.nc
sliced 1621-02-01 to 1622-01-17
   fixed time
   fixed longitude
sliced 1622-02-01 to 1623-01-17
   fixed time
   fixed longitude
sliced 1623-02-01 to 1624-01-17
   fixed time
   fixed longitude
sliced 1624-02-01 to 1625-01-17
   fixed time
   fixed longitude
sliced 1625-02-01 to 1626-01-17
   fixed time
   fixed longitude
sliced 1626-02-01 to 1627-01-17
   fixed time
   fixed longitude
sliced 1627-02-01 to 1628-01-17
   fixed time
   fixed longitude
sliced 1628-02-01 to 1629-01-17
   fixed time
   fixed longitude
sliced 1629-02-01 to 1630-01-17
   fixed time
   fixed longitude
sliced 1630-02-01 to 1631-01-17
   fixed time
   fixed longitude
sliced 1631-02-01 to 1632-01-17
   fixed time
   fixed longitude
sliced 1632-02-01 to 1633-01-17
   fixed time
   fixed longitude
sliced 1633-

HDF5-DIAG: Error detected in HDF5 (1.14.6) MPI-process 0:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/glade/work/gl

wrote data to disk
CPU times: user 1min 13s, sys: 5.93 s, total: 1min 19s
Wall time: 15min 47s


In [10]:
%%time
if var != 'RESTOM' and vert_lev[comp][var_ind]:
    filepath = path_to_arch+cesm2piC+'/'+comp+'/proc/tseries/'+time_path[comp][freq]+'/'
    num = 0
    slice_list = []
    for startyr in startyrs:
        yr_range = np.array([str(i).zfill(4) for i in np.arange(startyr,startyr+76)])
        
        endyr = startyr+74
        start_floor = np.floor_divide(startyr,100)
        end_floor = np.floor_divide(endyr,100)
        
        if start_floor == end_floor:
            if start_floor == 0:
                instart = str(1).zfill(4)
                inend = str(99).zfill(4)
            elif start_floor == 19:
                instart = str(1900).zfill(4)
                inend = str(2000).zfill(4)
            else:
                instart = str(start_floor*100).zfill(4)
                inend = str(start_floor*100+99).zfill(4)
        
            yr_extn = {'in': ["."+instart+"01-"+inend+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}

            
            # Open dataset
            filename = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr_extn['in'][freq]+'nc'
            print(filepath+filename)
            ds = xr.open_dataset(filepath+filename,chunks=chunks[comp])
        else:
            if start_floor == 0:
                instart1 = str(1).zfill(4)
                inend1 = str(99).zfill(4)
            else:
                instart1 = str(start_floor*100).zfill(4)
                inend1 = str(start_floor*100+99).zfill(4)
                
            if end_floor == 19:
                instart2 = str(1900).zfill(4)
                inend2 = str(2000).zfill(4)
            else:
                instart2 = str(end_floor*100).zfill(4)
                inend2 = str(end_floor*100+99).zfill(4)
    
            yr_extn1 = {'in': ["."+instart1+"01-"+inend1+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
    
            yr_extn2 = {'in': ["."+instart2+"01-"+inend2+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
            yr_extn = {'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}

            # Open dataset
            filename1 = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr_extn1['in'][freq]+'nc'
            filename2 = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr_extn2['in'][freq]+'nc'
            print(filepath+filename1)
            print(filepath+filename2)
            ds = xr.open_mfdataset([filepath+filename1, filepath+filename2],chunks=chunks[comp])
        
        
        dsv = ds
        if start_floor == end_floor:
            filename = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'PS'+yr_extn['in'][freq]+'nc'
            dsp = xr.open_dataset(filepath+filename,chunks=chunks[comp])
        else:
            filename1 = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'PS'+yr_extn1['in'][freq]+'nc'
            filename2 = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'PS'+yr_extn2['in'][freq]+'nc'
            dsp = xr.open_mfdataset([filepath+filename1, filepath+filename2],chunks=chunks[comp])
            
        dsv['PS'] = dsp['PS']
        #del ds
        
        processed_list = []
        for i in range(0,len(yr_range)-1):
            # Selection of final time
            final_time = pd.date_range(str(1950+i)+'-01-01',str(1950+i)+'-12-01',freq='MS')
                
            # Selection of data    
            startyr_sl = yr_range[i]
            endyr_sl = yr_range[i+1]
            ann_slice = dsv.sel(time=slice(startyr_sl+'-02-01',endyr_sl+'-01-17')) #if add_cyclic else dsv.sel(time=slice(startyr+'-02-10',endyr+'-01-17'),lat=slice(0,90))
            print('sliced '+startyr_sl+'-02-01 to '+endyr_sl+'-01-17')
            
            ann_slice = FixTime(ann_slice)
            ann_slice = ann_slice.assign_coords(time=final_time)
            print('   fixed time')

            ann_slice = FixLongitude(ann_slice, False)
            print('   fixed longitude')
            
            ann_slice = InterPlevels(ann_slice, var)
            print('   interpolated p-levels')
            processed_list.append(ann_slice)

        
        processed_out = xr.concat(processed_list,dim='time').chunk({'time':111})
        processed_out = processed_out.expand_dims(slice=[slice_numbers[num]])
        slice_list.append(processed_out)  
        print('concatenated slice '+str(num))
        num = num+1
        
    for i in range(0,len(slice_list)):
        if i == 0:
            slice_list[i].to_zarr(path_to_outdata+cesm2piC+h_ext[comp][freq]+var+'.195001-202412.zarr', 
                            group=var)
            print('saved initial zarr store')
        else:
            print('saving slice '+str(i+1))
            slice_list[i].to_zarr(path_to_outdata+cesm2piC+h_ext[comp][freq]+var+'.195001-202412.zarr', 
                                    append_dim='slice', mode='a-',group=var)
    print('wrote data to disk')
    
       

CPU times: user 5 μs, sys: 0 ns, total: 5 μs
Wall time: 8.11 μs


In [11]:
%%time

if var == 'RESTOM':
    filepath = path_to_arch+cesm2piC+'/'+comp+'/proc/tseries/'+time_path[comp][freq]+'/'
    num = 0
    for startyr in startyrs:
        yr_range = np.array([str(i).zfill(4) for i in np.arange(startyr,startyr+76)])
        
        endyr = startyr+74
        start_floor = np.floor_divide(startyr,100)
        end_floor = np.floor_divide(endyr,100)
        
        if start_floor == end_floor:
            if start_floor == 0:
                instart = str(1).zfill(4)
                inend = str(99).zfill(4)
            elif start_floor == 19:
                instart = str(1900).zfill(4)
                inend = str(2000).zfill(4)
            else:
                instart = str(start_floor*100).zfill(4)
                inend = str(start_floor*100+99).zfill(4)
        
            yr_extn = {'in': ["."+instart+"01-"+inend+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
        
            # Open dataset
            filename_fsnt = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'FSNT'+yr_extn['in'][freq]+'nc'
            filename_flnt = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'FLNT'+yr_extn['in'][freq]+'nc'
            print(filepath+filename_fsnt)
            ds_fsnt = dask.delayed(xr.open_dataset)(filepath+filename_fsnt,chunks=chunks[comp])
            ds_flnt = dask.delayed(xr.open_dataset)(filepath+filename_flnt,chunks=chunks[comp])

            ds_fsnt = ds_fsnt['FSNT']
            ds_flnt = ds_flnt['FLNT']

            dsv = ds_fsnt-ds_flnt
            dsv = dsv.rename(var) 
            
        else:
            if start_floor == 0:
                instart1 = str(1).zfill(4)
                inend1 = str(99).zfill(4)
            else:
                instart1 = str(start_floor*100).zfill(4)
                inend1 = str(start_floor*100+99).zfill(4)
                
            if end_floor == 19:
                instart2 = str(1900).zfill(4)
                inend2 = str(2000).zfill(4)
            else:
                instart2 = str(end_floor*100).zfill(4)
                inend2 = str(end_floor*100+99).zfill(4)
    
            yr_extn1 = {'in': ["."+instart1+"01-"+inend1+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
    
            yr_extn2 = {'in': ["."+instart2+"01-"+inend2+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
            yr_extn = {'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
    
            # Open dataset
            filename1_fsnt = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'FSNT'+yr_extn1['in'][freq]+'nc'
            filename2_fsnt = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'FSNT'+yr_extn2['in'][freq]+'nc'

            filename1_flnt = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'FLNT'+yr_extn1['in'][freq]+'nc'
            filename2_flnt = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'FLNT'+yr_extn2['in'][freq]+'nc'
            
            print(filepath+filename1_fsnt)
            print(filepath+filename2_fsnt)
            ds_fsnt = dask.delayed(xr.open_mfdataset)([filepath+filename1_fsnt, filepath+filename2_fsnt],chunks=chunks[comp])
            ds_flnt = dask.delayed(xr.open_mfdataset)([filepath+filename1_flnt, filepath+filename2_flnt],chunks=chunks[comp])

            ds_fsnt = ds_fsnt['FSNT']
            ds_flnt = ds_flnt['FLNT']

            dsv = ds_fsnt-ds_flnt
            dsv = dsv.rename(var)
        
        #del ds
        
        processed_list = []
        for i in range(0,len(yr_range)-1):
            # If monthly data
            if freq == 0:
                startyr = yr_range[i]
                endyr = yr_range[i+1]
                ann_slice = dsv.sel(time=slice(startyr+'-02-01',endyr+'-01-17')) #if add_cyclic else dsv.sel(time=slice(startyr+'-02-10',endyr+'-01-17'),lat=slice(0,90))
                print('sliced '+startyr+'-02-01 to '+endyr+'-01-17')
                
                fixedtime_data = dask.delayed(FixTime)(ann_slice)
                print('   fixed time')
        
                if comp == 'ice':
                    fixedgrid_data = dask.delayed(FixGrid)(fixedtime_data,'gx1v7')
                    processed_list.append(fixedgrid_data)
                    print('   fixed CICE grid')
        
                    #del ann_slice, fixedtime_data, fixedgrid_data
                elif comp == 'ocn':
                    processed_list.append(fixedtime_data)
        
                    #del ann_slice, fixedtime_data
                else:
                    fixedgrid_data = dask.delayed(FixLongitude)(fixedtime_data, False)
                    # If 3D data, interpolate to pressure levels
                    if vert_lev[comp][var_ind]:
                        addplev_data = dask.delayed(InterPlevels)(fixedgrid_data, var)
                        processed_list.append(addplev_data)
                    else:
                        processed_list.append(fixedgrid_data)
                    print('   fixed longitude')
        
                    #del ann_slice, fixedtime_data, fixedgrid_data
                    
                
            # If daily data
            else:
                startyr = yr_range[i]
                endyr = yr_range[i]
                ann_slice = dsv.sel(time=slice(startyr+'-01-01',endyr+'-12-31')) #if add_cyclic else dsv.sel(time=slice(startyr+'-01-01',endyr+'-12-31'), lat=slice(0,90))
                print('sliced '+startyr+'-01-01 to '+endyr+'-12-31')
        
                if comp == 'ice':
                    fixedgrid_data = dask.delayed(FixGrid)(ann_slice,'gx1v7')
                    processed_list.append(fixedgrid_data)
                    print('   fixed CICE grid')
        
                    #del ann_slice, fixedgrid_data
                elif comp == 'ocn':
                    processed_list.append(ann_slice)
        
                    #del ann_slice
                else:
                    fixedgrid_data = dask.delayed(FixLongitude)(ann_slice, False)
                    processed_list.append(fixedgrid_data)
                    print('   fixed longitude')
        
                    #del ann_slice, fixedgrid_data
        
        processed_comp = dask.compute(*processed_list)
        print('computed list')
        
        if not_vert and freq == 0:
            processed_out = xr.concat(processed_comp,dim='time').chunk({'time':111})
            print('concatenated data')
    
            processed_out = processed_out.expand_dims(slice=[slice_numbers[num]])
            
            processed_out.to_netcdf(path_to_outdata+cesm2piC+h_ext[comp][freq]+var+yr_extn['out'][freq]+'nc', 
                                        format='NETCDF4',encoding={var: {"zlib": True, "complevel": 1}})
            print('wrote data to disk')
    
        num = num+1

CPU times: user 5 μs, sys: 0 ns, total: 5 μs
Wall time: 7.63 μs


In [12]:
client.shutdown()